# Comparative Study of Pretrained CNN Models for Plant Disease Classification

This notebook performs a comparative analysis of **5 pretrained CNN models** using transfer learning on a plant leaf disease dataset.

**Models compared:**
1. MobileNetV2
2. ResNet50
3. VGG16
4. EfficientNetB0
5. DenseNet121

All models use the same dataset, training configuration, and evaluation metrics for a fair comparison.

---
## 1. Setup & Configuration

In [ ]:
# Mount Google Drive to access the dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Verify GPU is available (T4 recommended)
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU detected: {gpus[0].name}")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# All imports
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import (
    MobileNetV2,
    ResNet50,
    VGG16,
    EfficientNetB0,
    DenseNet121
)
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import time
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================
# CONFIGURATION - All hyperparameters in one place
# ============================================

# Paths for dataset (Drive and fast local copy)
DRIVE_DATASET = "/content/drive/MyDrive/Plant_leave_diseases_dataset_with_augmentation"
LOCAL_DATASET = "/content/dataset"

# Copy dataset from Drive to Colab local disk for much faster I/O
# We use shell cp -r instead of shutil.copytree because it is
# significantly faster and less likely to hang with many small files.
if not os.path.exists(LOCAL_DATASET):
    print("Copying dataset from Drive to local disk (this takes a few minutes, only once)...")
    # Remove any partial/old copy just in case
    !rm -rf /content/dataset
    # Fast recursive copy from Drive to local disk
    !cp -r "/content/drive/MyDrive/Plant_leave_diseases_dataset_with_augmentation" /content/dataset
    print("Done! Dataset copied to local disk.")
else:
    print("Dataset already exists on local disk, skipping copy.")

DATASET_DIR = LOCAL_DATASET

# Training parameters (kept identical for all models for fair comparison)
IMG_SIZE = (224, 224)
BATCH_SIZE = 24
EPOCHS = 5
LEARNING_RATE = 1e-4

# Models to compare
MODELS_TO_TRAIN = [
    "MobileNetV2",
    "ResNet50",
    "VGG16",
    "EfficientNetB0",
    "DenseNet121"
]

# Directory to save results
SAVE_DIR = "/content/drive/MyDrive/plant_disease_project/results"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Configuration set.")
print(f"  Dataset: {DATASET_DIR}")
print(f"  Image size: {IMG_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Models: {MODELS_TO_TRAIN}")

---
## 2. Load Dataset

In [ ]:
# Load training set (80%) and validation set (20%) from the same directory
# Using a fixed seed ensures the same split every time
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Extract class information
class_names = train_ds.class_names
num_classes = len(class_names)

print(f"\nDataset loaded successfully!")
print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names}")

# Optimize data loading with prefetching (overlaps loading with training)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
# Preview sample images from the dataset
plt.figure(figsize=(16, 10))
for images, labels in train_ds.take(1):
    for i in range(min(15, len(images))):
        ax = plt.subplot(3, 5, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]], fontsize=8)
        plt.axis("off")
plt.suptitle("Sample Images from Dataset", fontsize=16)
plt.tight_layout()
plt.show()

---
## 3. Model Architecture

Each model follows the same structure:
- **Pretrained backbone** (ImageNet weights, frozen)
- **Model-specific preprocessing** layer
- **Custom classification head**: GlobalAveragePooling2D → Dense(256) → Dropout(0.5) → Dense(num_classes)

In [ ]:
def create_model(model_name, num_classes, input_shape=(224, 224, 3)):
    """
    Creates a transfer learning model with the specified pretrained backbone.
    All models share the same classification head for fair comparison.
    """

    # Select the pretrained backbone and its preprocessing function
    backbone_config = {
        "MobileNetV2": {
            "model_fn": MobileNetV2,
            "preprocess": tf.keras.applications.mobilenet_v2.preprocess_input,
        },
        "ResNet50": {
            "model_fn": ResNet50,
            "preprocess": tf.keras.applications.resnet50.preprocess_input,
        },
        "VGG16": {
            "model_fn": VGG16,
            "preprocess": tf.keras.applications.vgg16.preprocess_input,
        },
        "EfficientNetB0": {
            "model_fn": EfficientNetB0,
            "preprocess": tf.keras.applications.efficientnet.preprocess_input,
        },
        "DenseNet121": {
            "model_fn": DenseNet121,
            "preprocess": tf.keras.applications.densenet.preprocess_input,
        },
    }

    config = backbone_config[model_name]

    # Load pretrained backbone without the top classification layer
    base_model = config["model_fn"](
        weights="imagenet",
        include_top=False,
        input_shape=input_shape
    )

    # Freeze all backbone layers (we only train the classification head)
    base_model.trainable = False

    # Build the full model
    inputs = layers.Input(shape=input_shape)

    # Apply model-specific preprocessing (each model expects different input scaling)
    x = config["preprocess"](inputs)

    # Pass through the frozen backbone
    x = base_model(x, training=False)

    # Custom classification head (identical for all models)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name=model_name)
    return model

print("create_model() defined.")

---
## 4. Training & Evaluation Functions

In [ ]:
def train_model(model, train_dataset, val_dataset, epochs=EPOCHS, lr=LEARNING_RATE):
    """
    Compiles and trains the model. Returns training history and elapsed time.
    All models use the same optimizer, loss, and training schedule.
    """
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    print(f"\nTraining {model.name}...")
    start_time = time.time()

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs
    )

    training_time = time.time() - start_time
    print(f"{model.name} training completed in {training_time:.1f} seconds ({training_time/60:.1f} min)")

    return history, training_time


def evaluate_model(model, val_dataset, class_names):
    """
    Runs predictions on the validation set and computes metrics.
    Returns true labels, predicted labels, and the confusion matrix.
    """
    y_true = []
    y_pred = []

    for images, labels in val_dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Calculate metrics using weighted average (accounts for class imbalance)
    accuracy = np.mean(y_true == y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    return {
        "y_true": y_true,
        "y_pred": y_pred,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "confusion_matrix": cm
    }

print("Training and evaluation functions defined.")

---
## 5. Visualization Functions

In [ ]:
def plot_training_history(history, model_name):
    """Plots accuracy and loss curves for a single model."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train', marker='o')
    axes[0].plot(history.history['val_accuracy'], label='Validation', marker='s')
    axes[0].set_title(f'{model_name} - Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss
    axes[1].plot(history.history['loss'], label='Train', marker='o')
    axes[1].plot(history.history['val_loss'], label='Validation', marker='s')
    axes[1].set_title(f'{model_name} - Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def generate_confusion_matrix(cm, class_names, model_name):
    """Plots a normalized confusion matrix heatmap."""
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(18, 15))
    sns.heatmap(
        cm_normalized,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.title(f"Confusion Matrix - {model_name}", fontsize=16)
    plt.xlabel("Predicted", fontsize=12)
    plt.ylabel("True", fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


def generate_classification_report(y_true, y_pred, class_names, model_name):
    """Prints a detailed per-class classification report."""
    print(f"\n{'='*60}")
    print(f"Classification Report - {model_name}")
    print(f"{'='*60}")
    print(classification_report(
        y_true, y_pred,
        target_names=class_names,
        zero_division=0,
        digits=4
    ))

print("Visualization functions defined.")

---
## 6. Train All Models

This is the main training loop. Each model is built, trained, and evaluated sequentially using identical settings.

In [ ]:
# Storage for results across all models
all_histories = {}    # Training history for each model
all_results = {}      # Evaluation metrics for each model
all_times = {}        # Training time for each model
comparison_rows = []  # For the final comparison DataFrame

for model_name in MODELS_TO_TRAIN:
    print(f"\n{'#'*60}")
    print(f"  MODEL: {model_name}")
    print(f"{'#'*60}")

    # Step 1: Build model
    model = create_model(model_name, num_classes)
    total_params = model.count_params()
    print(f"Total parameters: {total_params:,}")

    # Step 2: Train model
    history, training_time = train_model(model, train_ds, val_ds)
    all_histories[model_name] = history
    all_times[model_name] = training_time

    # Step 3: Evaluate model
    print(f"\nEvaluating {model_name}...")
    results = evaluate_model(model, val_ds, class_names)
    all_results[model_name] = results

    # Step 4: Plot training curves for this model
    plot_training_history(history, model_name)

    # Step 5: Store metrics for comparison table
    comparison_rows.append({
        "Model": model_name,
        "Val Accuracy": f"{results['accuracy']:.4f}",
        "Precision": f"{results['precision']:.4f}",
        "Recall": f"{results['recall']:.4f}",
        "F1 Score": f"{results['f1_score']:.4f}",
        "Training Time (s)": f"{training_time:.1f}",
        "Parameters": f"{total_params:,}"
    })

    print(f"\n{model_name} -> Val Accuracy: {results['accuracy']:.4f} | F1: {results['f1_score']:.4f}")

    # Free memory before loading next model
    del model
    tf.keras.backend.clear_session()

print(f"\n{'='*60}")
print("ALL MODELS TRAINED SUCCESSFULLY!")
print(f"{'='*60}")

---
## 7. Comparison Table

In [ ]:
# Create a comparison DataFrame
comparison_df = pd.DataFrame(comparison_rows)
print("\n" + "="*80)
print("          COMPARATIVE STUDY - MODEL PERFORMANCE SUMMARY")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Save the comparison table as CSV
comparison_df.to_csv(os.path.join(SAVE_DIR, "comparison_table.csv"), index=False)
print(f"\nComparison table saved to: {SAVE_DIR}/comparison_table.csv")

---
## 8. Comparison Visualizations

In [ ]:
# Plot 1: Validation Accuracy vs Epochs for all models
plt.figure(figsize=(12, 6))
for model_name, history in all_histories.items():
    plt.plot(history.history['val_accuracy'], label=model_name, marker='o')

plt.title('Validation Accuracy Comparison Across Models', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "accuracy_comparison.png"), dpi=150)
plt.show()

In [ ]:
# Plot 2: Validation Loss vs Epochs for all models
plt.figure(figsize=(12, 6))
for model_name, history in all_histories.items():
    plt.plot(history.history['val_loss'], label=model_name, marker='o')

plt.title('Validation Loss Comparison Across Models', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Loss', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "loss_comparison.png"), dpi=150)
plt.show()

In [ ]:
# Plot 3: Bar chart comparing final validation accuracy
model_names = list(all_results.keys())
accuracies = [all_results[m]['accuracy'] for m in model_names]
f1_scores = [all_results[m]['f1_score'] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, f1_scores, width, label='F1 Score', color='coral')

ax.set_title('Model Comparison - Accuracy & F1 Score', fontsize=16)
ax.set_ylabel('Score', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

# Add value labels on top of each bar
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "bar_chart_comparison.png"), dpi=150)
plt.show()

In [ ]:
# Plot 4: Training time comparison
times = [all_times[m] / 60 for m in model_names]  # Convert to minutes

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(model_names, times, color='mediumseagreen')
ax.set_xlabel('Training Time (minutes)', fontsize=12)
ax.set_title('Training Time Comparison', fontsize=16)
ax.grid(axis='x', alpha=0.3)

for bar, t in zip(bars, times):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2.,
            f'{t:.1f} min', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "training_time_comparison.png"), dpi=150)
plt.show()

---
## 9. Detailed Evaluation - Confusion Matrices

In [ ]:
# Generate confusion matrix for each model
for model_name in MODELS_TO_TRAIN:
    results = all_results[model_name]
    generate_confusion_matrix(results['confusion_matrix'], class_names, model_name)

---
## 10. Detailed Evaluation - Classification Reports

In [ ]:
# Print classification report for each model
for model_name in MODELS_TO_TRAIN:
    results = all_results[model_name]
    generate_classification_report(results['y_true'], results['y_pred'], class_names, model_name)

---
## 11. Save Results

In [ ]:
# Save the final comparison table again with a summary
print("\n" + "="*80)
print("          FINAL RESULTS SUMMARY")
print("="*80)
print(comparison_df.to_string(index=False))

# Find the best model
best_idx = np.argmax([all_results[m]['accuracy'] for m in MODELS_TO_TRAIN])
best_model = MODELS_TO_TRAIN[best_idx]
best_acc = all_results[best_model]['accuracy']

print(f"\nBest performing model: {best_model} (Accuracy: {best_acc:.4f})")
print(f"\nAll results saved to: {SAVE_DIR}")
print("  - comparison_table.csv")
print("  - accuracy_comparison.png")
print("  - loss_comparison.png")
print("  - bar_chart_comparison.png")
print("  - training_time_comparison.png")